# 02 Ablation & Interaction Effects

Observed two-factor interaction analysis.

In [ ]:
import os
from pathlib import Path

os.environ.setdefault("POLARS_SKIP_CPU_CHECK", "1")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import scipy.stats as stats

assets_dir = Path("../paper_assets")
assets_dir.mkdir(exist_ok=True)

df = pd.DataFrame(pl.read_parquet("../results/aggregated.parquet").to_dicts())
df = df[df["model"] == "deepseek-v4-flash"]
for column in ["tool_count", "top_k", "experiment", "run_id"]:
    if column not in df.columns:
        df[column] = pd.NA

def map_config_features(config_name):
    name = (config_name or "").lower()
    hierarchical = "a_flat" not in name
    workflow = any(token in name for token in ["d_hier", "d_min", "d_wf", "e_with", "f_full", "four_in_one"])
    return hierarchical, workflow

features = df["config_name"].apply(map_config_features)
df["hierarchical"] = [item[0] for item in features]
df["workflow"] = [item[1] for item in features]

def diagnostic_plot(filename, title, message):
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.axis("off")
    ax.text(0.5, 0.62, title, ha="center", va="center", fontsize=14, weight="bold")
    ax.text(0.5, 0.42, message, ha="center", va="center", wrap=True)
    fig.savefig(assets_dir / filename)
    plt.show()

print(f"Loaded {len(df)} traces from aggregated.parquet")
df.head()

## Interaction Heatmap

In [ ]:
df_main = df[df["experiment"] == "phase4_batch"]
pivot_strict = df_main.groupby(["hierarchical", "workflow"])["strict_success"].mean().unstack().reindex(index=[False, True], columns=[False, True])
pivot_func = df_main.groupby(["hierarchical", "workflow"])["functional_success"].mean().unstack().reindex(index=[False, True], columns=[False, True])
pivot_weighted = df_main.groupby(["hierarchical", "workflow"])["weighted_success"].mean().unstack().reindex(index=[False, True], columns=[False, True])
values = np.ma.masked_invalid(pivot_func.to_numpy(dtype=float))
cmap = plt.cm.Blues.copy()
cmap.set_bad("#eeeeee")

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(values, cmap=cmap, vmin=0, vmax=1)
fig.colorbar(im, ax=ax, label="Functional Success Rate")
ax.set_xticks(np.arange(2))
ax.set_yticks(np.arange(2))
ax.set_xticklabels(["False", "True"])
ax.set_yticklabels(["False", "True"])
for i in range(2):
    for j in range(2):
        val_strict = pivot_strict.iloc[i, j]
        val_func = pivot_func.iloc[i, j]
        val_weighted = pivot_weighted.iloc[i, j]
        if pd.isna(val_strict) or pd.isna(val_func) or pd.isna(val_weighted):
            text = "N/A"
        else:
            text = f"Strict: {val_strict:.3f}\nFunc: {val_func:.3f}\nWeighted: {val_weighted:.3f}"
        ax.text(j, i, text, ha="center", va="center", color="white" if pd.notna(val_func) and val_func > 0.5 else "black")
ax.set_xlabel("Workflow Enabled")
ax.set_ylabel("Hierarchical Architecture")
ax.set_title("Interaction Effect of Architecture and Workflow")
plt.savefig(assets_dir / "h6_interaction_heatmap.png")
plt.show()

complete_cells = pivot_func.notna().sum().sum()
print(f"Observed interaction cells: {complete_cells}/4")
if complete_cells < 4:
    print("ANOVA is not recommended until all four cells have observed data.")